# Trassenfinder API Query

Collects the training dataset for the night train energy model from Deutsche
Bahn's Trassenfinder API: energy consumption, route distance and technical
travel time for every route segment, queried once per standard train
composition.

Runs top to bottom — no manual steps, no cells to skip.

> **The API client lives in `trassenfinder.py`**, shared with
> `01b_speed_sweep.ipynb`. The payload template, the `mutter` resolution and
> the error handling all moved there so the two collections cannot drift apart.
> `build_payload()` emits a byte-identical body to the version that produced
> `samples_all.csv`, so **this notebook does not need re-running** because of
> that refactor.

## Two route sources

| Source | Routes | What it is |
|---|---|---|
| `ontd` | 95 | Real German night-train segments from the ONTD workbook — station to station, as actually operated |
| `synthetic` | 100 | Generated point-to-point pairs, sampled for geographic and length diversity |

They cover different parts of the design space. ONTD segments are mostly short
(median 78 km air distance) because real night trains stop frequently; the
generated set is mostly long (median 355 km) and spreads across N-S, E-W and
diagonal axes. Running both lets us check whether coefficients fitted on
operational segments hold on arbitrary station pairs — if they diverge, the
model is picking up something about how night trains are routed rather than
about the physics of moving a train.

Each source is collected and saved separately, then combined with a `source`
column so the regression notebook can fit on either or both.

## What this notebook cannot tell you

Every request here uses the composition's own `v_max`, either 200 or 230 km/h,
and both sit above line speed on essentially every path. So the booked speed
never binds and **speed is never varied**: about 99% of the variation in
average speed across this sample is a property of the line, not of the train.
No speed coefficient can be estimated from it. That is what
`01b_speed_sweep.ipynb` is for.

## Pipeline

1. Load compositions and both route sources, normalise to a common schema
2. Confirm the request template
3. Confirm the query function
4. Resolve every station once (pre-flight)
5. Collect each source
6. Save per source, plus a combined file
7. Compare the two samples and quality check

## API

Deutsche Bahn Trassenfinder, `POST /api/web/routen/suche`. No authentication.
Stations are identified by DS100 code.

**Runtime:** roughly 1,560 collection requests plus around 150 for the
pre-flight. Budget 30–50 minutes.

## 1. Setup

Both sources are normalised to the same five columns — `route_name`,
`start_stop_name`, `start_ds100`, `end_stop_name`, `end_ds100` — so everything
downstream is source-agnostic.

Two corrections are applied before querying:

- **`DS100_CORRECTIONS`** — codes Trassenfinder rejects in both mother and child
  form. `KKSU` for Köln Süd is not a valid Betriebsstelle; the correct code is
  `KKS`. Add here rather than editing the source CSVs, so the correction stays
  visible and upstream fixes drop in without conflict.
- **Segments without a DS100** — Lörrach Autoreisezug Terminal has no code in
  the ONTD export and cannot be queried.

In [ ]:
import pandas as pd

import trassenfinder as tf
from data_sources import DATA_DIR, SEED_DIR, SOURCES_DIR, source_input

SOURCES_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
SEED_DIR.mkdir(parents=True, exist_ok=True)

# DS100 codes that Trassenfinder rejects in both mother and child form.
DS100_CORRECTIONS = {
    "KKSU": "KKS",  # Köln Süd
}

SCHEMA = ["route_name", "start_stop_name", "start_ds100", "end_stop_name", "end_ds100"]


def load_ontd():
    """Real night-train segments — already in the target schema."""
    df = pd.read_csv(source_input("routes_ontd.csv"))
    return df[SCHEMA]


def load_synthetic():
    """Generated station pairs — renamed into the target schema.

    The file has no route identifier, so one is minted per row; segment_id
    downstream needs it to be unique.
    """
    df = pd.read_csv(source_input("routes_synthetic.csv"))

    df = df.rename(
        columns={
            "start_DS100": "start_ds100",
            "end_DS100": "end_ds100",
            "start_name": "start_stop_name",
            "end_name": "end_stop_name",
        }
    )
    df["route_name"] = [f"SYN-{i:03d}" for i in range(1, len(df) + 1)]

    return df[SCHEMA]


SOURCES = {
    "ontd": load_ontd,
    "synthetic": load_synthetic,
}

compositions = pd.read_csv(source_input("compositions.csv"))

route_sets = {}
for name, loader in SOURCES.items():
    df = loader()
    df = df.replace({"start_ds100": DS100_CORRECTIONS, "end_ds100": DS100_CORRECTIONS})

    dropped = df[df[["start_ds100", "end_ds100"]].isna().any(axis=1)]
    df = df.dropna(subset=["start_ds100", "end_ds100"]).reset_index(drop=True)
    route_sets[name] = df

    print(f"{name:10s} {len(df):4d} routes  ({len(dropped)} dropped, no DS100)")
    for _, row in dropped.iterrows():
        print(f"           - {row['start_stop_name']} -> {row['end_stop_name']}")

print("\nCompositions:", len(compositions))
print("Total requests:", sum(len(df) for df in route_sets.values()) * len(compositions))

# Station pairs present in both sources would be queried twice with identical
# settings, so the duplicate is worth knowing about before the run.
pairs = {
    name: set(zip(df["start_ds100"], df["end_ds100"]))
    for name, df in route_sets.items()
}
overlap = pairs["ontd"] & pairs["synthetic"]
print("Overlapping station pairs:", len(overlap), sorted(overlap) if overlap else "")

# Braking and line class are derived per composition, not fixed in the template.
# Printed here because a wrong value caps the achievable speed and would bound
# what this whole collection can say.
print("\nDerived per composition:")
for _, comp in compositions.iterrows():
    brh = tf.bremshundertstel(comp)
    print(f"  {comp['composition_id']:14s} {brh:4d} BrH   {tf.streckenklasse(comp)}")
    assert brh > tf.BRH_MG_DEDUCTION_THRESHOLD, (
        f"{comp['composition_id']} at {brh} BrH is below the "
        f"{tf.BRH_MG_DEDUCTION_THRESHOLD} threshold: 8 t per R+Mg vehicle must "
        "be deducted, which bremshundertstel() does not model."
    )

## 2. Request payload template

`trassenfinder.PAYLOAD_TEMPLATE` — passenger long-distance traffic with a
locomotive (`spfv_lok`). Route and composition fields are overwritten per
request; everything else is held constant across both sources, and across the
speed sweep, so that energy differences are attributable to route and train,
not to search settings.

Three settings shape the result and matter when interpreting the data:

- `schnellfahrstrecken_meiden` — high-speed lines avoided, as night trains
  generally do not use them (night-time maintenance windows)
- `gewichtung_parameter` — the route returned is a 40/30/30 compromise between
  distance, time and energy, not the shortest path
- `bremshundertstel`, `streckenklasse`, `bremsstellung` — fixed for all
  compositions, so they act as a level effect rather than a difference between
  trains

The cell below only reports it. To change a setting, edit the module — that way
the sweep changes with it instead of quietly diverging.

In [ ]:
settings = tf.PAYLOAD_TEMPLATE["sucheinstellungen"]
zug = tf.PAYLOAD_TEMPLATE["wegpunkte"][0]["zugcharakteristik"]

print("Endpoint:      ", tf.URL)
print("Verkehrsart:   ", settings["verkehrsart"])
print("Infrastruktur: ", tf.PAYLOAD_TEMPLATE["infrastruktur_id"])
print("Weighting:     ", settings["gewichtung_parameter"])
print("Streckenklasse:", zug["streckenklasse"], " Bremshundertstel:",
      zug["bremshundertstel"], " Bremsstellung:", zug["bremsstellung"])
print("Avoidance:")
for key, value in settings["vermeidung_parameter"].items():
    if value:
        print("   ", key)

# Overwritten per request: hauptnummer, wagenzuglaenge_m, wagenzugmasse_t,
# wagenanzahl, v_max, and both betriebsstelle blocks.
print("\nPayload template ready")

## 3. Query function

`trassenfinder.query()`. `mutter` is a fixed property of each Betriebsstelle,
not a free choice: Trassenfinder rejects a Mutterbetriebsstelle sent as a child
and a child sent as a mother, both with HTTP 400. Köln Hbf (`KK`) is only valid
as a mother; Aachen Hbf (`KA`), Koblenz (`KKO`), Freiburg (`RF`), Hanau
(`FH  N`), Erfurt (`UE  P`) and Berlin Hbf (`BLS`) only as children.

The flag is therefore resolved per station and cached across both sources —
and across the speed sweep, since the cache lives on the module.

`TrassenfinderError` carries the response body. `raise_for_status()` would
discard it, and that body is the only thing that says which of the two stations
was rejected.

In [ ]:
# The composition's own v_max is used unless one is passed explicitly, which is
# what 01b does. Nothing here overrides it.
example = tf.build_payload("AH", "MH", compositions.iloc[0], True, True)
example_zug = example["wegpunkte"][0]["zugcharakteristik"]

print("Example request for", compositions.iloc[0]["composition_id"])
print("  hauptnummer:     ", example_zug["triebfahrzeug"]["hauptnummer"])
print("  wagenzugmasse_t: ", example_zug["wagenzugmasse_t"])
print("  wagenzuglaenge_m:", example_zug["wagenzuglaenge_m"])
print("  wagenanzahl:     ", example_zug["wagenanzahl"])
print("  v_max:           ", example_zug["v_max"])

# wagenzugmasse_t is the trailing mass: coaches at 80% load, WITHOUT the
# locomotive, which is specified separately by hauptnummer. Everything fitted
# on this data is therefore per tonne of trailing mass — see section 2 of 02.
print("\nQuery function ready")

## 4. Station pre-flight

Every unique DS100 across both sources is probed once against a fixed reference
station to determine its `mutter` flag, before any collection starts. Two
reasons to do this separately: an invalid code costs one request instead of
eight, and the collection loops then run with a warm cache and no retries.

Stations that resolve neither way are not Betriebsstellen in this
infrastructure version. Their routes are **dropped here**, not carried into the
collection: without a `mutter` flag every one of their requests falls back to
the default and returns HTTP 400, which wastes eight calls per route and buries
the genuine routing failures in the failure table.

The generated route list contains seven such codes: `MURB`, `RBSS`, `BSAL`,
`ADT`, `AOH`, `BWSS`, `AKWS` — Umrathshausen, Baiersbronn Schule, Berlin
Schönhauser Allee, Hamburg Diebsteich, Hamburg-Othmarschen, Berlin Wannsee and
Hamburg Kornweg. Almost all are S-Bahn or local stops that the station export's
name-keyword filter failed to catch, and Trassenfinder does not carry them as
Betriebsstellen in this infrastructure version. They are not correctable
through `DS100_CORRECTIONS`; the fix is to remove them from
`sources/routes_synthetic.csv`.

In [ ]:
probe_composition = compositions.iloc[0]

stations = sorted(
    {s for df in route_sets.values() for s in df["start_ds100"]}
    | {s for df in route_sets.values() for s in df["end_ds100"]}
)

invalid_stations = tf.resolve_stations(stations, probe_composition)

print("Stations resolved:", len(tf.MUTTER_CACHE), "/", len(stations))
print(
    "Mother:",
    sum(tf.MUTTER_CACHE.values()),
    " Child:",
    len(tf.MUTTER_CACHE) - sum(tf.MUTTER_CACHE.values()),
)

if invalid_stations:
    # Drop rather than warn: an unresolved station has no mutter flag, so every
    # one of its requests would fall back to the default and 400. Eight wasted
    # calls per route, and a failure table that hides the real routing failures.
    print("\nINVALID:", invalid_stations)

    for name, df in route_sets.items():
        keep = ~(
            df["start_ds100"].isin(invalid_stations)
            | df["end_ds100"].isin(invalid_stations)
        )
        if (~keep).any():
            print(f"  {name}: dropping {(~keep).sum()} of {len(df)} routes")
            for _, row in df[~keep].iterrows():
                print(
                    f"    - {row['start_ds100']} -> {row['end_ds100']}  "
                    f"({row['start_stop_name']} -> {row['end_stop_name']})"
                )
        route_sets[name] = df[keep].reset_index(drop=True)

    print(
        "\nThese DS100 codes are not Betriebsstellen in this infrastructure "
        "version. Correct them in DS100_CORRECTIONS, or remove them from the "
        "route list under sources/."
    )
    print(
        "Remaining requests:",
        sum(len(df) for df in route_sets.values()) * len(compositions),
    )
else:
    print("\nAll stations valid")

## 5. Collect

Each source is queried independently: every route with every composition.
Failures are logged and skipped so one bad route cannot stop the run.

The ONTD set collects clean — every segment is operated, so it is routable by
construction. The generated set is not: roughly 40 of its 100 pairs return
`Es konnte keine Route … gefunden werden`, concentrated on regional stations
that a `D4` locomotive-hauled train cannot reach under these avoidance
settings. Those are logged, not fatal, and the surviving pairs are the ones
worth having.

In [ ]:
def collect(routes, source, compositions):
    """Query every route in one source with every composition.

    Returns:
        (results, failures) as two lists of dictionaries.
    """
    results = []
    failures = []

    print(f"[{source}] {len(routes) * len(compositions)} requests")

    for route_index, (_, route) in enumerate(routes.iterrows(), start=1):
        for _, composition in compositions.iterrows():
            try:
                result = tf.query(
                    route["start_ds100"], route["end_ds100"], composition
                )

                results.append(
                    {
                        "source": source,
                        "route_name": route["route_name"],
                        "start_stop_name": route["start_stop_name"],
                        "start_ds100": route["start_ds100"],
                        "end_stop_name": route["end_stop_name"],
                        "end_ds100": route["end_ds100"],
                        "composition_id": composition["composition_id"],
                        "n_coaches": composition["n_coaches"],
                        "weight_t": composition[
                            "coaches_gross_weight_80pct_t_wagenzugmasse"
                        ],
                        "length_m": composition["coaches_length_m_wagenzuglaenge"],
                        "v_max_kmh": composition["v_max_kmh"],
                        "bremshundertstel": tf.bremshundertstel(composition),
                        "streckenklasse": tf.streckenklasse(composition),
                        "energy_kwh": result["energy_kwh"],
                        "energy_components_kwh": result[
                            "energy_components_kwh"
                        ],
                        "energy_traktion_kwh": result["energy_traktion_kwh"],
                        "energy_hilfsbetriebe_kwh": result[
                            "energy_hilfsbetriebe_kwh"
                        ],
                        "energy_wagen_kwh": result["energy_wagen_kwh"],
                        "distance_km": result["distance_km"],
                        "travel_time_min": result["travel_time_min"],
                        "trassenpreis_eur": result["trassenpreis_eur"],
                        "stationspreis_eur": result["stationspreis_eur"],
                    }
                )

            except (tf.TrassenfinderError, ValueError) as e:
                failures.append(
                    {
                        "source": source,
                        "route_name": route["route_name"],
                        "start_ds100": route["start_ds100"],
                        "end_ds100": route["end_ds100"],
                        "composition_id": composition["composition_id"],
                        "error": str(e),
                    }
                )

        if route_index % 10 == 0 or route_index == len(routes):
            print(
                f"  {route_index}/{len(routes)} | ok: {len(results)} | "
                f"failed: {len(failures)}"
            )

    return results, failures


collected = {}
for name, df in route_sets.items():
    collected[name] = collect(df, name, compositions)
    print()

for name, (results, failures) in collected.items():
    print(f"{name:10s} ok: {len(results):4d}  failed: {len(failures):3d}")
    for failure in failures[:3]:
        print(
            f"           {failure['start_ds100']} -> {failure['end_ds100']} | "
            f"{failure['error'][:100]}"
        )

## 6. Save

Each source gets its own pair of files, and both are concatenated into a
combined dataset carrying the `source` column.

`samples_ontd.csv` is the primary training set. Fitting on the generated
pairs or on both means pointing at `samples_synthetic.csv` or
`samples_all.csv`.

In [ ]:
FILENAMES = {
    "ontd": ("samples_ontd.csv", "failures_ontd.csv"),
    "synthetic": ("samples_synthetic.csv", "failures_synthetic.csv"),
}

frames = []
for name, (results, failures) in collected.items():
    results_file, failures_file = FILENAMES[name]

    results_df = pd.DataFrame(results)
    pd.DataFrame(failures).to_csv(DATA_DIR / failures_file, index=False)
    results_df.to_csv(DATA_DIR / results_file, index=False)
    frames.append(results_df)

    print(f"✓ {name}: {len(results_df)} samples -> {results_file}")

combined = pd.concat(frames, ignore_index=True)
combined.to_csv(DATA_DIR / "samples_all.csv", index=False)

print(f"✓ combined: {len(combined)} samples -> samples_all.csv")

## 7. Compare the two samples

The question this run exists to answer: do the two sources describe the same
relationship, or does the ONTD set carry structure specific to how night trains
are routed?

Two things to look at. **Coverage** — the generated set extends the distance
range upward, where ONTD is thin: ONTD reaches 817 km with 80 samples above
400 km, the generated pairs reach 941 km with 216. **Energy intensity at matched
distance** — if kWh/km per band agrees, the samples are interchangeable and can
be pooled. A systematic offset would mean route character, not distance, is
driving part of the fit.

As collected on 2026-08-24 they agree: fleet-weighted kWh/km per composition
matches within 1% across all eight, and a source indicator on the per-km term
is not significant. The samples are poolable.

**Read the generated sample's survivors with care.** Routes fail at very
different rates by station category — 22% for long-distance to long-distance,
58% mixed, 71% regional to regional — because a locomotive-hauled train under
these settings frequently cannot reach branch-line stations at all. `D4` line
class and `knotenbahnhoefe_meiden` are the likely constraints. The surviving
sample is therefore biased towards main-line pairs, which is the part of the
network a night train uses, but it is a selection and not a random draw.

In [ ]:
combined["segment_id"] = (
    combined["route_name"].astype(str)
    + "__"
    + combined["start_ds100"].astype(str)
    + "__"
    + combined["end_ds100"].astype(str)
)
combined["avg_speed_kmh"] = combined["distance_km"] / (combined["travel_time_min"] / 60)
combined["kwh_per_km"] = combined["energy_kwh"] / combined["distance_km"]
combined["band"] = pd.cut(
    combined["distance_km"], [0, 25, 50, 100, 200, 400, 900, 2000]
)

print("Routes and coverage")
print(
    combined.groupby("source")
    .agg(
        samples=("energy_kwh", "size"),
        routes=("segment_id", "nunique"),
        d_min=("distance_km", "min"),
        d_median=("distance_km", "median"),
        d_max=("distance_km", "max"),
        speed_mean=("avg_speed_kmh", "mean"),
    )
    .round(1)
    .to_string()
)

print("\nMedian kWh/km by distance band and source")
print(
    combined.pivot_table(
        index="band",
        columns="source",
        values="kwh_per_km",
        aggfunc="median",
        observed=True,
    )
    .round(2)
    .to_string()
)

print("\nSamples per band and source")
print(
    combined.pivot_table(
        index="band",
        columns="source",
        values="energy_kwh",
        aggfunc="size",
        observed=True,
    ).to_string()
)

print("\nMedian avg speed by band and source")
print(
    combined.pivot_table(
        index="band",
        columns="source",
        values="avg_speed_kmh",
        aggfunc="median",
        observed=True,
    )
    .round(1)
    .to_string()
)

## 8. Quality check

Read back from disk, so this checks the files the regression notebook will
actually load.

In [ ]:
checks = pd.read_csv(DATA_DIR / "samples_all.csv")

checks["segment_id"] = (
    checks["route_name"].astype(str)
    + "__"
    + checks["start_ds100"].astype(str)
    + "__"
    + checks["end_ds100"].astype(str)
)

print("Samples:     ", len(checks))
print("Routes:      ", checks["segment_id"].nunique())
print("Compositions:", checks["composition_id"].nunique())
print("Duplicates:  ", checks.duplicated(["segment_id", "composition_id"]).sum())

missing = checks.isna().sum()
print("Missing:     ", int(missing.sum()))
if missing.sum():
    # Named per column, because a bare count says nothing about whether it
    # matters. A gap in trassenpreis_eur is cosmetic; one in energy_kwh is not.
    print(missing[missing > 0].to_string())

print("\nCompositions per route (want all 8):")
print(
    checks.groupby("segment_id")["composition_id"].nunique().value_counts().to_string()
)

print("\nRanges:")
print(
    checks[["distance_km", "weight_t", "energy_kwh", "travel_time_min"]]
    .describe()
    .round(1)
    .to_string()
)

# --- energy split reconciliation --------------------------------------------
# The per-route-point energy fields are CUMULATIVE running totals, so the value
# taken is the last point, never the sum. This check exists because summing them
# looked entirely reasonable and produced numbers 22x too large at the median
# with no error anywhere - the only visible symptom was an auxiliary share of
# 606%.
parts = (
    checks["energy_traktion_kwh"]
    + checks["energy_hilfsbetriebe_kwh"]
    + checks["energy_wagen_kwh"]
)

print("\nComponents sum vs their own reported total, % of trip energy:")
print(
    ((parts - checks["energy_components_kwh"]) / checks["energy_kwh"] * 100)
    .describe()
    .round(3)
    .to_string()
)

print("\nComponents total vs zusammenfassung total, % difference:")
divergence = (
    (checks["energy_components_kwh"] - checks["energy_kwh"]) / checks["energy_kwh"] * 100
)
print(divergence.describe().round(3).to_string())

if divergence.abs().median() > 0.5:
    print(
        "\n  The components and the summary disagree. Regeneration netted off "
        "one and not the other is the likeliest cause. Establish which is "
        "which before 02 fits anything: the split is only usable if it "
        "reconciles with the total the backend will price."
    )
else:
    print("\n  Split reconciles. 02 can fit traction and auxiliaries separately.")

aux_share = (
    checks["energy_hilfsbetriebe_kwh"].sum() / checks["energy_kwh"].sum() * 100
)
print(f"\nAuxiliaries: {aux_share:.1f}% of total energy")
print(
    "  energy_wagen_kwh all zero:",
    (checks["energy_wagen_kwh"] == 0).all(),
    "(coach hotel load is switched off in the request)",
)

print("\nFleet-weighted kWh/km by composition and source:")
print(
    checks.groupby(["source", "composition_id"])
    .apply(
        lambda d: d["energy_kwh"].sum() / d["distance_km"].sum(), include_groups=False
    )
    .unstack(0)
    .round(2)
    .to_string()
)

print(
    f"\nFleet total: "
    f"{checks['energy_kwh'].sum() / checks['distance_km'].sum():.2f} kWh/km"
)

## 9. Handover

### Generated files

| File | Contents |
|---|---|
| `calib/data/samples_ontd.csv` | ONTD night-train segments — the primary training set |
| `calib/data/samples_synthetic.csv` | Generated station pairs |
| `calib/data/samples_all.csv` | Both, with a `source` column |
| `calib/data/failures_*.csv` | Requests that could not be completed, with the API message |

Upload the `samples_*.csv` files to the Drive folder
(`ENERGY_DRIVE_FOLDER_ID`) after a run — they cannot be rebuilt at seed time.

### Columns

`source`, `route_name`, `start_stop_name`, `start_ds100`, `end_stop_name`,
`end_ds100`, `composition_id`, `n_coaches`, `weight_t`, `length_m`,
`v_max_kmh`, `bremshundertstel`, `streckenklasse`, `energy_kwh`,
`energy_traktion_kwh`, `energy_hilfsbetriebe_kwh`, `energy_wagen_kwh`,
`distance_km`, `travel_time_min`, `trassenpreis_eur`, `stationspreis_eur`

Everything from `energy_kwh` onward comes from Trassenfinder;
`bremshundertstel` and `streckenklasse` are derived per composition and sent
with the request; the rest are carried through from the input files.

**The energy split is the reason to prefer this sample over anything collected
before 2026-08-30.** Traction scales with the square of speed, auxiliaries with
time. Collected as one total, a speed coefficient has to absorb both and comes
out wrong at the slow end of the range.

### Known limitations

- **Germany only.** No terrain variation in either source, so no terrain
  coefficient can be estimated. Austrian and Swiss routes are needed for that,
  and neither of these sources provides them.
- **Speed is never varied.** Both `v_max` values in use sit above line speed, so
  the booked speed does not bind and average speed is a property of the line
  rather than of the train. Run `01b_speed_sweep.ipynb` for that dimension.
- **Coach hotel load is excluded.** The template sets
  `zusaetzlicher_energieverbrauch_pro_wagen_kw` to 0, so `energy_wagen_kwh`
  comes back at zero. For a night train with sleeping cars that is a real
  omission: heating, air conditioning and lighting run all night. Setting it is
  a modelling decision that has not been taken, not an oversight to patch
  silently.
- **`travel_time_min` is technical running time** — no dwell, no recovery
  margin — and reflects Trassenfinder's own route choice, not the backend's
  routing engine.
- **`weight_t` is the trailing mass**, coaches at 80% load without the
  locomotive, and a composition-level constant. Eight compositions, eight
  distinct weights, near-collinear with coach count and train length, so weight
  cannot be separated from other composition attributes.
- **Routes, not country legs.** The backend predicts per country leg; both
  sources are station to station.
- **The generated set is not operational.** Its pairs were sampled for
  diversity, not drawn from timetables, so it should be treated as a robustness
  check on the ONTD fit rather than as evidence about real night-train routing.

### Reproducing

Restart the kernel and run all cells. Collection is idempotent in the sense
that it overwrites all output CSVs, but **not** against the API: Trassenfinder's
network state moves between infrastructure versions, so two runs months apart
will not agree exactly.

Downstream: `01b_speed_sweep.ipynb`, then `02_energy_calibration.ipynb`.